In [1]:
import sys
import os

from PySide6.QtWidgets  import (QApplication,QMainWindow,QLabel, QScrollArea,
                                QPushButton,QLineEdit,QMessageBox,
                                QWidget,QVBoxLayout,QDialog,QVBoxLayout,QHBoxLayout,QStackedWidget)


from PySide6.QtCore                    import QTimer
from gui_design.Controller             import Controller, Stage
from gui_design.pure_functions         import counter
from gui_design.DataManager            import DataManager
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg
from PySide6.QtWidgets import QStyle



In [2]:
gui qt

In [6]:
class PlotWindow(QDialog):
    def __init__(self, figure, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Experiments so far")
        self.resize(800, 600)
        
        layout = QVBoxLayout(self)
        self.canvas = FigureCanvasQTAgg(figure)
        self.canvas.draw()
        layout.addWidget(self.canvas)

class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Experiment Control Center")
        self.resize(1100, 700)
        
        # Central widget of the main window
        central_widget = QWidget()
        self.setCentralWidget(central_widget)
        
        # Main horizontal layout: Sidebar on left, Stacked content on right
        main_layout = QHBoxLayout(central_widget)
        main_layout.setContentsMargins(0, 0, 0, 0)
        main_layout.setSpacing(0)
        
        # -----------------------------------------------------------------
        # 1. LEFT SIDEBAR NAVIGATION PANEL
        # -----------------------------------------------------------------
        sidebar_widget = QWidget()
        sidebar_widget.setObjectName("sidebar")
        sidebar_widget.setFixedWidth(220)
        sidebar_layout = QVBoxLayout(sidebar_widget)
        sidebar_layout.setContentsMargins(15, 20, 15, 20)
        sidebar_layout.setSpacing(10)
        
        # Create navigation buttons
        self.btn_dashboard = QPushButton("🏠  Dashboard")
        self.btn_experiment = QPushButton("🧪  Experiment")
        self.btn_plots = QPushButton("📊  Data & Plots")
        self.btn_logs = QPushButton("📄  Logs")
        self.btn_settings = QPushButton("⚙️  Settings")
        
        nav_buttons = [
            self.btn_dashboard, self.btn_experiment, 
            self.btn_plots, self.btn_logs, self.btn_settings
        ]
        
        for btn in nav_buttons:
            btn.setCheckable(True)
            sidebar_layout.addWidget(btn)
            
        sidebar_layout.addSpacing(15)

        # (1) Quick menu option below the main navigation buttons (identical quick actions)
        quick_label = QLabel("Quick Actions")
        quick_label.setStyleSheet("color: #64748b; font-size: 11px; font-weight: bold;")
        sidebar_layout.addWidget(quick_label)

        self.quick_start_btn = QPushButton("▶ Start")
        self.quick_start_btn.setObjectName("start_button")
        self.quick_stop_btn = QPushButton("⏹ Stop")
        self.quick_stop_btn.setObjectName("stop_button")
        self.quick_reset_btn = QPushButton("🔄 Reset")
        self.quick_reset_btn.setObjectName("reset_button")
        self.quick_quit_btn = QPushButton("🚪 Quit")
        self.quick_quit_btn.setObjectName("quit_button")

        quick_actions = [self.quick_start_btn, self.quick_stop_btn, self.quick_reset_btn, self.quick_quit_btn]
        for qbtn in quick_actions:
            sidebar_layout.addWidget(qbtn)

        # Connect quick buttons to the same command functions
        self.quick_start_btn.clicked.connect(self.start_command)
        self.quick_stop_btn.clicked.connect(self.stop_command)
        self.quick_reset_btn.clicked.connect(self.reset_command)
        self.quick_quit_btn.clicked.connect(self.close)

        sidebar_layout.addStretch()
        
        # -----------------------------------------------------------------
        # 2. RIGHT SIDE: STACKED WIDGET (CONTENT VIEWS)
        # -----------------------------------------------------------------
        self.stack = QStackedWidget()
        
        # Create your separate view pages
        self.dashboard_page = QWidget()
        self.setup_dashboard_view()
        
        self.experiment_page = QWidget()
        self.setup_experiment_view()  
        
        self.plots_page = QWidget()
        self.setup_plots_view()

        self.logs_page = QWidget()
        self.setup_logs_view()

        self.settings_page = QWidget()
        self.setup_settings_view()
        
        # Add pages to the stack
        self.stack.addWidget(self.dashboard_page)  # Index 0
        self.stack.addWidget(self.experiment_page) # Index 1
        self.stack.addWidget(self.plots_page)      # Index 2
        self.stack.addWidget(self.logs_page)       # Index 3
        self.stack.addWidget(self.settings_page)   # Index 4
        
        # Connect sidebar buttons to switch stack indices
        self.btn_dashboard.clicked.connect(lambda: self.switch_page(0))
        self.btn_experiment.clicked.connect(lambda: self.switch_page(1))
        self.btn_plots.clicked.connect(lambda: self.switch_page(2))
        self.btn_logs.clicked.connect(lambda: self.switch_page(3))
        self.btn_settings.clicked.connect(lambda: self.switch_page(4))
        
        # Default selection (Dashboard)
        self.btn_dashboard.setChecked(True)
        self.stack.setCurrentIndex(0)
        
        # Add both to main window layout
        main_layout.addWidget(sidebar_widget)
        main_layout.addWidget(self.stack)

        # --------------------------------------------------
        # Constants & Internal variables
        # --------------------------------------------------
        self.OMNI = "Omni_stage"
        self.MODEL = "MODEL_stage"
        self.omni_filename = "expttsd"
        self.MODEL_filename = "MODEL"

        self.variables_set = False
        self.process_stop = False
        self.CONTROLLER = None
        self.DATAPROCESSOR = None
        self.omni_run_count = 0
        self.MODEL_run_count = 0
        self.initial_exp_count = 0
        self.omni_folder = ""
        self.model_folder = ""

        # Apply general stylesheet
        self.create_style()

        # --------------------------------------------------
        # Timer
        # --------------------------------------------------
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.monitoring_loop)
        self.monitor_mode = None
        self.monitor_path = None
        self.monitor_num = None

    def switch_page(self, index):
        """Switches the active page in the stack and manages button states."""
        self.stack.setCurrentIndex(index)
        
        buttons = [self.btn_dashboard, self.btn_experiment, self.btn_plots, self.btn_logs, self.btn_settings]
        
        for i, btn in enumerate(buttons):
            if i < self.stack.count():
                btn.setChecked(i == index)

    # --- Page layout builders ---
    def setup_dashboard_view(self):
        layout = QVBoxLayout(self.dashboard_page)
        layout.addWidget(QLabel("<h1>Dashboard View</h1><p>Main control panel widgets go here.</p>"))

    def setup_plots_view(self):
        """(2) Set up Data & Plots page to hold embedding canvases dynamically."""
        layout = QVBoxLayout(self.plots_page)
        layout.setContentsMargins(20, 20, 20, 20)
        
        title_label = QLabel("Data & Plots View")
        title_label.setStyleSheet("font-size: 18px; font-weight: bold; color: #f8fafc;")
        layout.addWidget(title_label)

        # Container layout or placeholder layout area for the generated plot canvas
        self.plot_container_layout = QVBoxLayout()
        
        self.plot_placeholder_label = QLabel("No experiment data plotted yet. Click 'Plot data' to load.")
        self.plot_placeholder_label.setStyleSheet("color: #64748b; font-size: 13px;")
        self.plot_container_layout.addWidget(self.plot_placeholder_label)
        
        layout.addLayout(self.plot_container_layout)

    def setup_logs_view(self):
        layout = QVBoxLayout(self.logs_page)
        layout.addWidget(QLabel("<h1>Logs</h1><p>Log info plotted here.</p>"))

    def setup_settings_view(self):
        layout = QVBoxLayout(self.settings_page)
        layout.addWidget(QLabel("<h1>Setting View</h1><p>Experiment and modeling settings.</p>"))

    def setup_experiment_view(self):
        # 1. Create a QScrollArea
        scroll_area = QScrollArea(self.experiment_page)
        scroll_area.setWidgetResizable(True)
        scroll_area.setStyleSheet("QScrollArea { border: none; background-color: transparent; }")

        # 2. Create an inner widget to hold all your layout controls
        scroll_content = QWidget()

        # 3. Instantiate your old widgets, attaching them to scroll_content instead of self.experiment_page
        self.status_label = QLabel("", scroll_content)
        self.error_label = QLabel("", scroll_content)
        
        self.exp_path_input = QLineEdit(scroll_content)
        self.exp_path_input.setPlaceholderText("OMNI folder path")
        self.MODEL_path_input = QLineEdit(scroll_content)
        self.MODEL_path_input.setPlaceholderText("MODEL folder path")
        self.exp_number_input = QLineEdit(scroll_content)
        self.exp_number_input.setPlaceholderText("No. experiments")

        self.omni_label = QLabel("OMNIBUS", scroll_content)
        self.MODEL_label = QLabel("MODEL", scroll_content)
        self.current_label = QLabel("Currently running:", scroll_content)         
        self.variables_label = QLabel("" , scroll_content)
        self.omni_count_label = QLabel("0" , scroll_content)
        self.model_count_label = QLabel("0" , scroll_content)

        self.define_button = QPushButton("Define settings", scroll_content)
        self.define_button.setObjectName("define_button")
        self.start_button = QPushButton("Start automated experiments", scroll_content)
        self.start_button.setObjectName("start_button")         
        self.stop_button = QPushButton("Stop", scroll_content)
        self.stop_button.setObjectName("stop_button")         
        self.reset_button = QPushButton("Reset", scroll_content)
        self.reset_button.setObjectName("reset_button")         
        self.plot_button = QPushButton("Plot data", scroll_content)
        self.plot_button.setObjectName("plot_button")         
        self.quit_button = QPushButton("Quit", scroll_content)
        self.quit_button.setObjectName("quit_button")         

        # Connections & Icons
        self.define_button.clicked.connect(self.define_command)
        self.start_button.clicked.connect(self.start_command)
        self.stop_button.clicked.connect(self.stop_command)
        self.reset_button.clicked.connect(self.reset_command)
        self.plot_button.clicked.connect(self.plot_expts)
        self.quit_button.clicked.connect(self.close)

        self.start_button.setIcon(self.style().standardIcon(QStyle.StandardPixmap.SP_MediaPlay))
        self.stop_button.setIcon(self.style().standardIcon(QStyle.StandardPixmap.SP_MediaStop))

        # Absolute geometry positions inside the scrollable content container
        self.status_label.setGeometry(30, 20, 300, 25)
        self.error_label.setGeometry(30, 45, 300, 25)
        
        self.exp_path_input.setGeometry(30, 110, 280, 35)
        self.MODEL_path_input.setGeometry(30, 160, 280, 35)
        self.exp_number_input.setGeometry(30, 210, 120, 35)
        self.define_button.setGeometry(30, 265, 390, 45)

        self.current_label.setGeometry(450, 110, 180, 25)
        self.omni_label.setGeometry(450, 150, 90, 25)
        self.MODEL_label.setGeometry(560, 150, 90, 25)
        self.omni_count_label.setGeometry(450, 185, 90, 30)
        self.model_count_label.setGeometry(560, 185, 90, 30)
        self.variables_label.setGeometry(450, 235, 200, 25)

        self.start_button.setGeometry(30, 365, 185, 45)
        self.stop_button.setGeometry(235, 365, 185, 45)
        self.reset_button.setGeometry(30, 420, 185, 45)
        self.plot_button.setGeometry(235, 420, 185, 45)
        self.quit_button.setGeometry(30, 485, 390, 40)

        # 4. Set scroll widget content layout bounds
        scroll_content.setMinimumSize(700, 560)
        scroll_area.setWidget(scroll_content)

        # 5. Pack the scroll area into the main experiment page view
        page_layout = QVBoxLayout(self.experiment_page)
        page_layout.setContentsMargins(0, 0, 0, 0)
        page_layout.addWidget(scroll_area)

    # ======================================================
    # STYLE
    # ======================================================

    def create_style(self):
        self.setStyleSheet("""
            QWidget {background-color: #0b0f19; color: #f8fafc; font-family: 'Segoe UI', Arial, sans-serif;}
            QPushButton {background-color: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 8px; font-weight: 600; padding: 6px 12px;}
            QPushButton#start_button {background-color: #166534; border: 1px solid #22c55e;}
            QPushButton#start_button:hover {background-color: #15803d;}
            QPushButton#stop_button {background-color: #991b1b; border: 1px solid #ef4444;}
            QPushButton#stop_button:hover {background-color: #b91c1c;}
            QPushButton#define_button {background: qlineargradient(x1:0, y1:0, x2:1, y2:0, stop:0 #4f46e5, stop:1 #7c3aed); border: none;}
            QPushButton#define_button:hover {background: qlineargradient(x1:0, y1:0, x2:1, y2:0, stop:0 #4338ca, stop:1 #6d28d9);}
            QPushButton:hover {background-color: #334155;}
            QPushButton:pressed {background-color: #475569;}
            QLineEdit {background-color: #111827; color: #f8fafc; border: 1px solid #374151; border-radius: 6px; padding: 6px 10px;}
            QLineEdit:focus {border: 1px solid #6366f1;}
            QLabel {color: #94a3b8; font-size: 13px;}
            QGroupBox {border: 1px solid #1e293b; border-radius: 10px; margin-top: 12px; font-weight: bold; background-color: #0f172a; padding-top: 15px;}
            QGroupBox::title {subcontrol-position: top left; left: 15px; padding: 0 5px; color: #f8fafc;}
        """)

    # ======================================================
    # HELPERS & COMMANDS (Unchanged Logic)
    # ======================================================

    def upon_omni_finished(self):
        self.omni_label.setStyleSheet("color: grey;")    
        self.MODEL_label.setStyleSheet("color: green;")    
        self.omni_run_count = (self.CONTROLLER.omni_count)    
        self.omni_count_label.setText(str(self.omni_run_count))
     
    def upon_MODEL_finished(self):
        self.omni_label.setStyleSheet("color: green;")
        self.MODEL_label.setStyleSheet("color: grey;")
        self.MODEL_run_count = (self.CONTROLLER.model_count)
        self.model_count_label.setText(str(self.MODEL_run_count))

    def monitoring_MODEL(self, path, num):
        if self.CONTROLLER.process_stop:
            return True
    
        self.CONTROLLER.current_stage = Stage.MODEL
        
        if self.CONTROLLER.if_model_finished(path):
            self.CONTROLLER.model_count += 1
            self.upon_MODEL_finished()
            PATH, exp_count = (self.CONTROLLER.pathcreator_omni(self.CONTROLLER.model_count, self.CONTROLLER.omni_folder))
            self.CONTROLLER.current_omni_folder = PATH
            self.monitor_mode = "OMNI"
            self.monitor_path = PATH
            self.monitor_num = exp_count
            self.timer.start(3000)
            return True
        else:
            self.omni_label.setStyleSheet("color: grey;")
            self.MODEL_label.setStyleSheet("color: green;")
            return False

    def monitoring_omni(self, path, num):
        if self.CONTROLLER.process_stop:
            return True
    
        self.CONTROLLER.current_stage = Stage.OMNI
   
        if self.CONTROLLER.if_omni_finished(path, num):
            PATH = self.CONTROLLER.pathcreator_model(self.CONTROLLER.omni_count, self.CONTROLLER.model_folder)
            self.CONTROLLER.current_model_folder = PATH
            self.CONTROLLER.omni_count += 1
            self.upon_omni_finished()

            self.DATAPROCESSOR.read_data_from = (self.CONTROLLER.current_omni_folder)
            self.DATAPROCESSOR.data_reader()

            self.monitor_mode = "MODEL"
            self.monitor_path = PATH
            self.timer.start(3000)
            return True
        else:
            self.omni_label.setStyleSheet("color: green;")
            self.MODEL_label.setStyleSheet("color: grey;")
            return False

    def set_buttons_enabled(self, enabled):
        buttons = [self.start_button, self.define_button, self.reset_button, self.quit_button, self.quick_start_btn, self.quick_reset_btn]
        for button in buttons:
            button.setEnabled(enabled)

    def monitoring_loop(self):
        if self.CONTROLLER is None:
            return
    
        if self.process_stop:
            self.timer.stop()
            return
    
        if self.monitor_mode == "OMNI":
            self.monitoring_omni(self.monitor_path, self.monitor_num)
        elif self.monitor_mode == "MODEL":
            self.monitoring_MODEL(self.monitor_path, self.monitor_num)

    def define_command(self):
        self.status_label.setText("")
        self.error_label.setText("")
        folder_loc = (self.exp_path_input.text().strip())
        folder_MODEL_loc = (self.MODEL_path_input.text().strip())
        no_exp = (self.exp_number_input.text().strip())

        if not folder_loc or not folder_MODEL_loc or not no_exp:
            self.status_label.setText("Please fill all fields")
            return

        if not os.path.isdir(folder_loc):
            self.error_label.setText("Invalid OMNI folder")
            return

        if not os.path.isdir(folder_MODEL_loc):
            self.error_label.setText("Invalid MODEL folder")
            return

        try:
            no_exp = int(no_exp)
            if no_exp <= 0:
                raise ValueError
        except ValueError:
            self.error_label.setText("Enter a positive integer")
            return

        self.omni_folder = folder_loc
        self.model_folder = folder_MODEL_loc
        self.initial_exp_count = no_exp
        self.CONTROLLER = Controller(self.omni_folder, self.model_folder, self.initial_exp_count, self.omni_filename, self.MODEL_filename)
        self.CONTROLLER.exp_num(no_exp)
        self.variables_set = True
        self.variables_label.setText("Variables set")
        
    def start_command(self):
        if not self.variables_set:
            QMessageBox.warning(self, "Error", "Please define variables first!")
            return

        if counter(self.omni_folder, self.omni_filename) != 0:
            QMessageBox.warning(self, "Error", "Files already in experiment folder")
            return
            
        if counter(self.model_folder, self.MODEL_filename) != 0:
            QMessageBox.warning(self, "Error", "Files already in MODEL folder")
            return
            
        self.set_buttons_enabled(False)
        self.CONTROLLER.process_stop = False
        self.DATAPROCESSOR = DataManager(self.omni_filename, self.MODEL_filename)
        self.monitor_mode = "OMNI"
        
        PATH, exp_count = (self.CONTROLLER.pathcreator_omni(self.CONTROLLER.model_count, self.CONTROLLER.omni_folder))
        self.CONTROLLER.current_omni_folder = PATH
        self.monitor_path = PATH
        self.monitor_num = exp_count
        self.CONTROLLER.start()
        
        self.timer.start(5000)

    def stop_command(self):
        if self.CONTROLLER:
            self.CONTROLLER.stop()
        self.timer.stop()
        self.set_buttons_enabled(True)

    def reset_command(self):
        self.variables_set = False
        self.CONTROLLER = None
        self.DATAPROCESSOR = None
        self.exp_path_input.clear()
        self.MODEL_path_input.clear()
        self.exp_number_input.clear()
        self.omni_count_label.setText("0")
        self.model_count_label.setText("0")
        self.status_label.setText("")
        self.error_label.setText("")
        self.variables_label.setText("")
        while self.plot_container_layout.count():
                item = self.plot_container_layout.takeAt(0)
                widget = item.widget()
                if widget:
                    widget.deleteLater()

    def plot_expts(self):
        if self.DATAPROCESSOR is None:
            QMessageBox.warning(self, "Error", "No data available yet!")
            return
 
        self.DATAPROCESSOR.read_data_from = (self.CONTROLLER.current_omni_folder)
        self.DATAPROCESSOR.data_reader()
        
        fig = self.DATAPROCESSOR.plot_generator()
        if fig:
            # (2) Embed/Plot the experiment plot directly onto the Data & Plots page container
            # Clear previous widgets/canvases in the plots layout if any exist
            while self.plot_container_layout.count():
                item = self.plot_container_layout.takeAt(0)
                widget = item.widget()
                if widget:
                    widget.deleteLater()

            self.canvas_embedded = FigureCanvasQTAgg(fig)
            self.canvas_embedded.draw()
            self.plot_container_layout.addWidget(self.canvas_embedded)

            # Optionally also open the separate popup window if desired, or switch directly to plots page view index (2)
            self.switch_page(2)

# ==========================================================
# MAIN
# ==========================================================

if __name__ == "__main__":
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)

    window = MainWindow()
    window.show()
    app.exec()